# MILK10k — Step 6a: Concept-level explainability (CPU)
Uses the saved embeddings (notebook 02) and out-of-fold probabilities (notebook 03). Every analysis is cross-fitted over the 5 folds.
1. **Concept fidelity** — can each image's embedding recover its MONET concepts? Can the *clinical* photo anticipate the *dermoscopic* concepts?
2. **Interpretable concept model** — how much of the diagnosis is explained by the 4 diagnostic concepts + metadata (vs the black-box model)? Global concept→diagnosis weights.
3. **Leave-one-input-out attribution** — which input drives each diagnosis (valid because the model was trained with inputs missing).
4. **Why was dermoscopy requested?** — per-lesion uncertainty about each dermoscopic concept, predicted from the clinical photo; validated against the actual dermoscopy gain.
5. **Shortcut audit** — do pen markings, hair or gel push predictions?
Grad-CAM (needs the GPU and the saved networks) is notebook 06b.

In [ ]:
import os, numpy as np, pandas as pd, warnings
warnings.filterwarnings("ignore")
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, balanced_accuracy_score, roc_auc_score
from scipy.stats import pearsonr, spearmanr

def find_file(name, tops=("/kaggle/input", "/kaggle/working")):
    for top in tops:
        for root, dirs, files in os.walk(top, followlinks=True):
            if name in files: return os.path.join(root, name)
    return None
EMB0 = find_file("emb_D_fold0.npy"); NPZ = find_file("oof_probs_fusion.npz"); FOLDS_CSV = find_file("milk10k_lesions_folds.csv")
assert EMB0 and NPZ and FOLDS_CSV, "attach notebook-o2 (embeddings, folds) and notebook-o3 (probabilities)"
STAGE2 = os.path.dirname(EMB0); OUT = "/kaggle/working/stage6"; os.makedirs(OUT, exist_ok=True)
les = pd.read_csv(FOLDS_CSV)
CLASSES = ['AKIEC','BCC','BEN_OTH','BKL','DF','INF','MAL_OTH','MEL','NV','SCCKA','VASC']; K = len(CLASSES)
ACTIONABLE = ["AKIEC", "BCC", "MAL_OTH", "MEL", "SCCKA"]; is_act_cls = np.isin(CLASSES, ACTIONABLE)
y = les["y"].values.astype(int); fold = les["fold"].values; n = len(y); y_act = is_act_cls[y]
DIAG = ["ulceration_crust", "vasculature_vessels", "erythema", "pigmented"]
ARTIF = ["hair", "gel_water_drop_fluid_dermoscopy_liquid", "skin_markings_pen_ink_purple_pen"]
SHORT = {"ulceration_crust": "ulceration/crust", "vasculature_vessels": "vessels", "erythema": "erythema",
         "pigmented": "pigmented", "hair": "hair", "gel_water_drop_fluid_dermoscopy_liquid": "gel/fluid",
         "skin_markings_pen_ink_purple_pen": "pen marking"}
def mon(c, m): return les[f"MONET_{c}_{m}"].values.astype(float)
site = pd.get_dummies(les["site"].fillna("unknown")).values.astype(float)
META = np.column_stack([les["age_approx"].fillna(les["age_approx"].median()).values, les["skin_tone_class"].values,
                        (les["sex"] == "male").values, site]).astype(float)
EMB = {(m, f): np.load(f"{STAGE2}/emb_{m}_fold{f}.npy").astype(np.float32) for m in "CD" for f in range(5)}
npz = np.load(NPZ); P = {p: npz[f"late_drop|{p}"].astype(float) for p in ["CDM", "CD", "CM", "DM", "C", "D", "M"]}
print("lesions", n, "| concepts", len(DIAG) + len(ARTIF))

## 1. Concept fidelity (cross-fitted ridge on the fold's embeddings; Pearson r on held-out lesions)

In [ ]:
def crossfit_ridge(Xfn, target, alpha=10.0):
    pred = np.zeros(n)
    for f in range(5):
        X = Xfn(f); tr, te = fold != f, fold == f
        sc = StandardScaler().fit(X[tr]); r = Ridge(alpha=alpha).fit(sc.transform(X[tr]), target[tr])
        pred[te] = r.predict(sc.transform(X[te]))
    return pred

rows = []
for c in DIAG + ARTIF:
    for src, tgt, Xfn in [("C", "C", lambda f: EMB[("C", f)]),
                          ("D", "D", lambda f: EMB[("D", f)]),
                          ("C+M", "D", lambda f: np.column_stack([EMB[("C", f)], META]))]:   # anticipate dermoscopic concept
        t = mon(c, tgt); p = crossfit_ridge(Xfn, t)
        rows.append(dict(concept=SHORT[c], kind="diagnostic" if c in DIAG else "artefact",
                         task=f"{src} image -> {tgt} concept", r=pearsonr(p, t)[0]))
fid = pd.DataFrame(rows).pivot_table(index=["kind", "concept"], columns="task", values="r").round(3)
fid.to_csv(f"{OUT}/concept_fidelity.csv"); fid

## 2. Interpretable concept model vs black box

In [ ]:
def concept_X(mods, with_meta=True):
    cols = [mon(c, m) for m in mods for c in DIAG]
    return np.column_stack(cols + ([META] if with_meta else []))

def crossfit_logreg(X):
    Pp = np.zeros((n, K))
    for f in range(5):
        tr, te = fold != f, fold == f
        sc = StandardScaler().fit(X[tr])
        clf = LogisticRegression(max_iter=3000, class_weight="balanced", C=1.0).fit(sc.transform(X[tr]), y[tr])
        tmp = np.zeros((te.sum(), K)); tmp[:, clf.classes_] = clf.predict_proba(sc.transform(X[te])); Pp[te] = tmp
    return Pp
def mf1(Pp): return f1_score(y, Pp.argmax(1), average="macro", labels=range(K), zero_division=0)
def auc(Pp): return roc_auc_score(y, Pp, multi_class="ovr", average="macro", labels=range(K))

rows = []
for name, mods, bb in [("C concepts + meta", ["C"], "CM"), ("D concepts + meta", ["D"], "DM"), ("C+D concepts + meta", ["C", "D"], "CDM")]:
    Pc = crossfit_logreg(concept_X(mods))
    rows.append(dict(model=name, concept_macro_f1=mf1(Pc), concept_auc=auc(Pc),
                     blackbox_pattern=bb, blackbox_macro_f1=mf1(P[bb]), blackbox_auc=auc(P[bb])))
cbm = pd.DataFrame(rows).set_index("model"); cbm.to_csv(f"{OUT}/concept_vs_blackbox.csv"); cbm.round(3)

In [ ]:
# Global explanation: standardised coefficients of the all-concepts model, fitted on all lesions (for display only)
X = concept_X(["C", "D"], with_meta=False); names = [f"{SHORT[c]} ({m})" for m in "CD" for c in DIAG]
clf = LogisticRegression(max_iter=3000, class_weight="balanced").fit(StandardScaler().fit_transform(X), y)
coef = pd.DataFrame(clf.coef_, index=[CLASSES[k] for k in clf.classes_], columns=names)
coef.to_csv(f"{OUT}/concept_coefficients.csv")
fig, ax = plt.subplots(figsize=(9, 5)); im = ax.imshow(coef.values, cmap="RdBu_r", vmin=-2, vmax=2, aspect="auto")
ax.set_xticks(range(len(names))); ax.set_xticklabels(names, rotation=45, ha="right"); ax.set_yticks(range(len(coef))); ax.set_yticklabels(coef.index)
plt.colorbar(im, label="standardised log-odds weight"); ax.set_title("Concept → diagnosis weights (C = clinical, D = dermoscopy)")
plt.tight_layout(); plt.savefig(f"{OUT}/fig_concept_weights.png", dpi=200); plt.show()

## 3. Leave-one-input-out attribution (drop in probability of the predicted class when an input is removed)

In [ ]:
pred = P["CDM"].argmax(1); idx = np.arange(n)
drop = {"clinical photo": P["CDM"][idx, pred] - P["DM"][idx, pred],
        "dermoscopy":     P["CDM"][idx, pred] - P["CM"][idx, pred],
        "metadata":       P["CDM"][idx, pred] - P["CD"][idx, pred]}
flip = {"clinical photo": P["DM"].argmax(1) != pred, "dermoscopy": P["CM"].argmax(1) != pred, "metadata": P["CD"].argmax(1) != pred}
correct = pred == y
att = pd.DataFrame({f"Δp removing {k}": pd.Series(v[correct]).groupby(y[correct]).mean() for k, v in drop.items()})
att = att.join(pd.DataFrame({f"flips if removing {k}": pd.Series(v[correct]).groupby(y[correct]).mean() for k, v in flip.items()}))
att["n_correct"] = pd.Series(y[correct]).value_counts().reindex(att.index).fillna(0).astype(int)
att.index = [CLASSES[k] for k in att.index]
att.to_csv(f"{OUT}/leave_one_input_out.csv"); att.round(3)

## 4. Why was dermoscopy requested? Concept uncertainty from the clinical photo
For each diagnostic concept, a ridge model predicts the **dermoscopic** concept from clinical embedding + metadata, and a second ridge model predicts the **size of that model's error** (trained on inner out-of-fold residuals — nested cross-fitting). High predicted error = "the clinical photo can't tell us what dermoscopy would show for this concept".

In [ ]:
def nested_uncertainty(target, alpha=10.0):
    mu, unc = np.zeros(n), np.zeros(n)
    for f in range(5):
        X = np.column_stack([EMB[("C", f)], META]); tr_idx, te = np.where(fold != f)[0], fold == f
        inner_res = np.zeros(len(tr_idx)); inner_fold = fold[tr_idx]
        for g in np.unique(inner_fold):                       # inner out-of-fold residuals
            a, b = tr_idx[inner_fold != g], tr_idx[inner_fold == g]
            sc = StandardScaler().fit(X[a]); r = Ridge(alpha=alpha).fit(sc.transform(X[a]), target[a])
            inner_res[inner_fold == g] = np.abs(target[b] - r.predict(sc.transform(X[b])))
        sc = StandardScaler().fit(X[tr_idx])
        mu[te] = Ridge(alpha=alpha).fit(sc.transform(X[tr_idx]), target[tr_idx]).predict(sc.transform(X[te]))
        unc[te] = Ridge(alpha=alpha).fit(sc.transform(X[tr_idx]), inner_res).predict(sc.transform(X[te]))
    return mu, unc
MU, UNC = {}, {}
for c in DIAG: MU[c], UNC[c] = nested_uncertainty(mon(c, "D"))
U = np.column_stack([UNC[c] for c in DIAG]); Uz = (U - U.mean(0)) / U.std(0)

eps = 1e-9
gain = np.log(P["CDM"][idx, y] + eps) - np.log(P["CM"][idx, y] + eps)
least_conf = 1 - P["CM"].max(1)
actual_err = np.column_stack([np.abs(mon(c, "D") - MU[c]) for c in DIAG])
val = pd.DataFrame({
    "Spearman(pred. uncertainty, actual concept error)": [spearmanr(U[:, j], actual_err[:, j])[0] for j in range(len(DIAG))],
    "Spearman(pred. uncertainty, dermoscopy gain)": [spearmanr(U[:, j], gain)[0] for j in range(len(DIAG))],
    "Spearman(pred. uncertainty, least-confidence)": [spearmanr(U[:, j], least_conf)[0] for j in range(len(DIAG))]},
    index=[SHORT[c] for c in DIAG])
val.loc["max over concepts (z)"] = [np.nan, spearmanr(Uz.max(1), gain)[0], spearmanr(Uz.max(1), least_conf)[0]]
val.to_csv(f"{OUT}/why_dermoscopy_validation.csv"); val.round(3)

In [ ]:
# Explanations for the lesions the least-confidence policy sends at a 30% rate
req = np.zeros(n, bool); req[np.argsort(-least_conf)[:int(0.3 * n)]] = True
top = Uz.argmax(1)
expl = pd.DataFrame({"lesion_id": les["lesion_id"], "true": [CLASSES[k] for k in y],
                     "CM_top2": [" / ".join(CLASSES[k] for k in np.argsort(-p)[:2]) for p in P["CM"]],
                     "most_uncertain_dermoscopic_concept": [SHORT[DIAG[j]] for j in top],
                     "predicted_value": [MU[DIAG[j]][i] for i, j in enumerate(top)],
                     "dermoscopy_gain": gain, "requested": req})
expl["explanation"] = ("Unsure between " + expl["CM_top2"] + "; the clinical photo cannot resolve dermoscopic "
                       + expl["most_uncertain_dermoscopic_concept"] + " -> dermoscopy recommended")
expl[req].to_csv(f"{OUT}/dermoscopy_request_explanations.csv", index=False)
print("Most frequent reason among requested lesions, by true class:")
pd.crosstab(expl.loc[req, "true"], expl.loc[req, "most_uncertain_dermoscopic_concept"], normalize="index").round(2)

## 5. Shortcut audit: do artefact concepts shift the model's 'actionable' probability?

In [ ]:
s_act = P["CDM"][:, is_act_cls].sum(1)
rng = np.random.RandomState(0); rows = []
for c in ARTIF:
    for m in "CD":
        v = mon(c, m); hi_ = v >= np.quantile(v, 0.75); lo_ = v <= np.quantile(v, 0.50)
        for grp, gm in [("benign lesions", ~y_act), ("actionable lesions", y_act)]:
            a, b = s_act[gm & hi_], s_act[gm & lo_]
            boots = [rng.choice(a, len(a)).mean() - rng.choice(b, len(b)).mean() for _ in range(1000)]
            rows.append(dict(artefact=f"{SHORT[c]} ({m})", within=grp, n_high=len(a),
                             diff_in_P_actionable=a.mean() - b.mean(),
                             ci_low=np.percentile(boots, 2.5), ci_high=np.percentile(boots, 97.5)))
audit = pd.DataFrame(rows).set_index(["artefact", "within"])
audit["flag"] = np.where((audit.ci_low > 0.05) | (audit.ci_high < -0.05), "CHECK", "")
audit.to_csv(f"{OUT}/shortcut_audit.csv"); audit.round(3)

In [ ]:
# Same question controlling for the true diagnosis: logistic model of the model's actionable call
from sklearn.linear_model import LogisticRegression
call = (s_act >= 0.5).astype(int)
Xa = np.column_stack([np.eye(K)[y]] + [StandardScaler().fit_transform(mon(c, m)[:, None]) for c in ARTIF for m in "CD"])
lr = LogisticRegression(max_iter=3000, C=10).fit(Xa, call)
art_coef = pd.Series(lr.coef_[0][K:], index=[f"{SHORT[c]} ({m})" for c in ARTIF for m in "CD"], name="log-odds per SD (adjusted for true class)")
art_coef.round(3)